In [2]:
import copy
from heapq import heappush, heappop
import time

# Có thể thay đổi n tùy ý (n=3, 4, 5,...)
n = 4

rows = [1, 0, -1, 0]
cols = [0, -1, 0, 1]

def calculate_manhattan_dist(mats, final_pos_map) -> int:
    """
    Tính khoảng cách Manhattan.
    Truyền vào map vị trí đích để tăng tốc độ tính toán (O(n^2)).
    """
    dist = 0
    for i in range(n):
        for j in range(n):
            val = mats[i][j]
            if val != 0:
                target_x, target_y = final_pos_map[val]
                dist += abs(i - target_x) + abs(j - target_y)
    return dist

def is_safe(x, y):
    return 0 <= x < n and 0 <= y < n

class Node:
    def __init__(self, parent, mats, empty_tile_pos, h_cost, level):
        self.parent = parent
        self.mats = mats
        self.empty_tile_pos = empty_tile_pos
        self.h_cost = h_cost
        self.level = level
        self.f_cost = h_cost + level # f(n) = g(n) + h(n)

    def __lt__(self, other):
        # Ưu tiên f_cost thấp, nếu f_cost bằng nhau thì ưu tiên h_cost thấp (nút gần đích hơn)
        if self.f_cost == other.f_cost:
            return self.h_cost < other.h_cost
        return self.f_cost < other.f_cost

def print_matrix(mats):
    for row in mats:
        print(" ".join(f"{val:3d}" for val in row))
    print("-" * (n * 4))

def print_path(root):
    if root is None: return
    print_path(root.parent)
    print_matrix(root.mats)

def solve(initial, final):
    # Tìm vị trí ô trống (0) trong ma trận khởi đầu
    empty_pos = None
    for i in range(n):
        for j in range(n):
            if initial[i][j] == 0:
                empty_pos = [i, j]
                break

    # Tạo map vị trí đích để tránh việc dùng vòng lặp lồng nhau trong mỗi bước tính h(n)
    final_pos_map = {}
    for i in range(n):
        for j in range(n):
            final_pos_map[final[i][j]] = (i, j)

    pq = []
    h = calculate_manhattan_dist(initial, final_pos_map)
    root = Node(None, initial, empty_pos, h, 0)
    heappush(pq, root)

    visited = set()
    start_time = time.time()

    while pq:
        current = heappop(pq)

        # Chuyển ma trận thành tuple của tuples để làm key trong set (tăng tốc độ hash)
        mats_tuple = tuple(map(tuple, current.mats))
        if mats_tuple in visited:
            continue
        visited.add(mats_tuple)

        if current.h_cost == 0:
            end_time = time.time()
            print_path(current)
            return

        for i in range(4):
            new_x, new_y = current.empty_tile_pos[0] + rows[i], current.empty_tile_pos[1] + cols[i]

            if is_safe(new_x, new_y):
                # Sao chép và hoán vị ô trống
                new_mats = [list(row) for row in current.mats] # Tốc độ nhanh hơn deepcopy
                x1, y1 = current.empty_tile_pos
                new_mats[x1][y1], new_mats[new_x][new_y] = new_mats[new_x][new_y], new_mats[x1][y1]

                h_child = calculate_manhattan_dist(new_mats, final_pos_map)
                child = Node(current, new_mats, [new_x, new_y], h_child, current.level + 1)
                heappush(pq, child)

# --- Khởi tạo dữ liệu mẫu cho n=4 (Có thể đổi n và ma trận) ---
initial_state = [
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 11, 12],
    [13, 0, 14, 15]
]

final_state = [
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 11, 12],
    [13, 14, 15, 0]
]

solve(initial_state, final_state)

  1   2   3   4
  5   6   7   8
  9  10  11  12
 13   0  14  15
----------------
  1   2   3   4
  5   6   7   8
  9  10  11  12
 13  14   0  15
----------------
  1   2   3   4
  5   6   7   8
  9  10  11  12
 13  14  15   0
----------------
